In [ ]:
!pip install geemap

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 7.3 MB/s eta 0:00:00


In [ ]:
import geemap
import ee


In [ ]:
ee.Authenticate()

In [ ]:
ee.Initialize(project='first-parser-394719')


In [ ]:
#for testing
Map = geemap.Map(center=[31.5204, 74.3587], zoom=9)
Map

Map(center=[31.5204, 74.3587], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDa…

In [ ]:
sentinel2 = ee.ImageCollection("COPERNICUS/S2_SR");

USDA_SoilData = ee.ImageCollection("NASA_USDA/HSL/SMAP10KM_soil_moisture");
# 30m Resolution for elevation
NASA_srtm = ee.Image("USGS/SRTMGL1_003");

#Modis
modis_lai_data = ee.ImageCollection('MODIS/061/MCD15A3H');
modis_vegetation_data = ee.ImageCollection("MODIS/061/MYD13Q1")

# Load Landsat 8 and 9 image collections
landsat8 = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
landsat9 = ee.ImageCollection("LANDSAT/LC09/C02/T1_L2")

# Load the sugarcane and other feature collections
sugarcane_tx = ee.FeatureCollection("projects/ee-sp20-bcs-003/assets/TX_intersection_5Y")

In [ ]:
sugarcane_tx.size().getInfo()

453

# Calculating Indices from Landset 8 and 9

In [ ]:
# Function to cloud mask image
def cloud_mask_landsat(image):
    qa = image.select('QA_PIXEL')
    cloud = 1 << 3
    cirrus = 1 << 9
    mask = qa.bitwiseAnd(cloud).eq(0).And(qa.bitwiseAnd(cirrus).eq(0))
    return image.updateMask(mask)

In [ ]:
# Function to calculate spectral indices
def calculate_indices_landsat(img):

    ndvi = img.normalizedDifference(['SR_B5', 'SR_B4']).rename('NDVI')
    gndvi = img.normalizedDifference(['SR_B5', 'SR_B3']).rename('GNDVI')
    evi = img.expression(
        '2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1))',
        {
            'NIR': img.select('SR_B5'),
            'RED': img.select('SR_B4'),
            'BLUE': img.select('SR_B2')
        }
    ).rename('EVI')
    nbr = img.normalizedDifference(['SR_B5', 'SR_B7']).rename('NBR')
    ndmi = img.normalizedDifference(['SR_B5', 'SR_B6']).rename('NDMI')
    ndwi = img.normalizedDifference(['SR_B3', 'SR_B5']).rename('NDWI')
    ndbi = img.normalizedDifference(['SR_B6', 'SR_B5']).rename('NDBI')
    ndbai = img.normalizedDifference(['SR_B6', 'SR_B7']).rename('NDBaI')
    mndwi = img.normalizedDifference(['SR_B3', 'SR_B6']).rename('MNDWI')
    return img.addBands([ndvi,gndvi,evi,nbr, ndmi, ndwi, ndbi, ndbai, mndwi])

In [ ]:
landsat8=landsat8.map(cloud_mask_landsat)
landsat8=landsat8.map(calculate_indices_landsat)

landsat9=landsat9.map(cloud_mask_landsat)
landsat9=landsat9.map(calculate_indices_landsat)

# Calculating indices from Sentinel 2 Setallite

In [ ]:
def sentinel2_bands_calculations(image):
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
    gndvi = image.normalizedDifference(['B8', 'B3']).rename('GNDVI')
    evi = image.expression(
        '2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1))',
        {
            'NIR': image.select('B8'),
            'RED': image.select('B4'),
            'BLUE': image.select('B2')
        }
    ).rename('EVI')
    red = image.select('B4')  # Red band (Sentinel-2 band 4)
    nir = image.select('B8')  # NIR band (Sentinel-2 band 8)
    savi = nir.subtract(red).divide(nir.add(red).add(0.5)).multiply(1.5).rename('SAVI')
    return image.addBands([ndvi, gndvi, evi, savi])

def mask_clouds(image):
    QA60 = image.select(['QA60'])
    cloud_bit_mask = 1 << 10
    cirrus_bit_mask = 1 << 11
    mask = QA60.bitwiseAnd(cloud_bit_mask).eq(0).And(QA60.bitwiseAnd(cirrus_bit_mask).eq(0))
    return image.updateMask(mask)

def calculate_indices(image):
    # Calculate VCI (Vegetation Condition Index)
    VCI = image.expression('(NDVI - NDVI_min) / (NDVI_max - NDVI_min) * 100',
                           {'NDVI': image.select('NDVI'),
                            'NDVI_min': 0,
                            'NDVI_max': 0.8}).rename('VCI')

    # Calculate Transformed Vegetation Index (TVI)
    TVI = image.expression('0.5 * (NIR - RED) / (NIR + RED + 0.5)',
                           {'NIR': image.select('B8'),
                            'RED': image.select('B4')}).rename('TVI')

    # Calculate Brightness Index (BI)
    BI = image.expression('sqrt((RED**2) + (NIR**2))',
                          {'RED': image.select('B4'),
                           'NIR': image.select('B8')}).rename('BI')

    # Calculate Second Brightness Index (BI2)
    BI2 = image.expression('(RED + NIR) / 2',
                           {'RED': image.select('B4'),
                            'NIR': image.select('B8')}).rename('BI2')

    # Calculate Color Index (CI)
    CI = image.expression('(NIR / RED) - 1',
                          {'NIR': image.select('B8'),
                           'RED': image.select('B4')}).rename('CI')

    # Calculate Clay Index (CI1)
    CI1 = image.expression('(RED / NIR) - 1',
                           {'RED': image.select('B4'),
                            'NIR': image.select('B8')}).rename('CI1')

    # Calculate SATVI (Soil Adjusted Total Vegetation Index)
    SATVI = image.expression('(NIR - RED - 0.5) / (NIR + RED + 0.5)',
                             {'NIR': image.select('B8'),
                              'RED': image.select('B4')}).rename('SATVI')

    # Calculate HVSI (Hue, Value, and Intensity)
    HVSI = image.expression('sqrt((NIR - RED)**2 + (NIR - GREEN)*(RED - GREEN))',
                            {'NIR': image.select('B8'),
                             'RED': image.select('B4'),
                             'GREEN': image.select('B3')}).rename('HVSI')

    # Calculate SOCI (Soil Organic Carbon Index)
    SOCI = image.expression('(NIR / RED) * (1 + RED - GREEN)',
                            {'NIR': image.select('B8'),
                             'RED': image.select('B4'),
                             'GREEN': image.select('B3')}).rename('SOCI')

    # Calculate ASI (Agricultural Stress Index)
    ASI = image.expression('RED - (GREEN + BLUE) / 2',
                           {'RED': image.select('B4'),
                            'GREEN': image.select('B3'),
                            'BLUE': image.select('B2')}).rename('ASI')

    # Calculate BSI (Bare Soil Index)
    BSI = image.expression('1 - (NIR + 0.05) / (RED + 0.05)',
                           {'NIR': image.select('B8'),
                            'RED': image.select('B4')}).rename('BSI')

    # Calculate MSAVI (Modified Soil Adjusted Vegetation Index)
    MSAVI = image.expression('(2 * NIR + 1 - sqrt((2 * NIR + 1)**2 - 8 * (NIR - RED))) / 2',
                             {'NIR': image.select('B8'),
                              'RED': image.select('B4')}).rename('MSAVI')

    return image.addBands([VCI, TVI, BI, BI2, CI, CI1, SATVI, HVSI, SOCI, ASI, BSI, MSAVI])


sentinel2 = sentinel2.map(mask_clouds)
sentinel2 = sentinel2.map(sentinel2_bands_calculations)
sentinel2 = sentinel2.map(calculate_indices)

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
# Splitting 43000 into sets so it can be used
total_count = sugarcane_la.size().getInfo()

# Calculate the number of features in each split
split_count = int(total_count / 10)

# Create a list to store the splits
sugarcane_sets = []

last_index = 0
# Iterate over 10 partitions
for i in range(1, 11):
    # Calculate the starting and ending index for each split
    start_index = (i - 1) * split_count
    end_index = i * split_count


    # Filter the FeatureCollection to get the current split
    split = sugarcane_la.toList(split_count, start_index)
    sugarcane_sets.append(split)


NameError: name 'sugarcane_la' is not defined

In [ ]:
def normalize_modis_value(x, min_val=-2000, max_val=10000, new_min=-1, new_max=1):
    normalized_value = ((x - min_val) / (max_val - min_val)) * (new_max - new_min) + new_min
    return normalized_value

In [ ]:
collection_years = [


    {
        'start_date': '2018-01-01',
        'end_date': '2018-12-31',
        'year':2018,
    },

    {
        'start_date': '2019-01-01',
        'end_date': '2019-12-31',
        'year':2019,
    },

    {
        'start_date': '2021-01-01',
        'end_date': '2021-12-31',
        'year':2021,
    },

    {
        'start_date': '2022-01-01',
        'end_date': '2022-12-31',
        'year':2022,
    }
]

In [ ]:
feature_data = []
index = 1
for year in collection_years:

  start_date = year['start_date']
  end_date = year['end_date']

  sentinel2_data = sentinel2 \
  .filterDate(start_date, end_date)


  landsat8_data = landsat8 \
  .filterDate(start_date, end_date)

  # landsat9_data = landsat9 \
  # .filterDate(start_date, end_date)

  modis_lai = modis_lai_data \
  .filterDate(start_date, end_date)

  modis_vegetation = modis_vegetation_data \
  .filterDate(start_date, end_date)

  USDA_SoilData_data = USDA_SoilData \
  .filterDate(start_date, end_date)

  landsat8_imagery = landsat8_data.mosaic().select(['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7','NBR','NDVI','GNDVI','EVI','NDMI','NDWI','NDBI','NDBaI','MNDWI'])
  # landsat9_imagery = landsat9_data.mosaic().select(['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7','NBR','NDVI','GNDVI','EVI','NDMI','NDWI','NDBI','NDBaI','MNDWI'])
  sentinel_2_imagery = sentinel2_data.median().select(['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B9', 'B11', 'B12', 'NDVI', 'EVI', 'GNDVI','SAVI','VCI', 'TVI', 'BI', 'BI2', 'CI', 'CI1', 'SATVI', 'HVSI', 'SOCI', 'ASI', 'BSI', 'MSAVI'])
  modis_lai_imagery = modis_lai.mosaic().select(['Lai'])
  modis_vegetation_imagery = modis_vegetation.mosaic().select('NDVI','EVI')
  USDA_imagery = USDA_SoilData_data.mosaic().select(['ssm'])



  field_id = 1

  for set in sugarcane_sets:

    table = set.getInfo()
    for record in table:

      polygon = ee.Feature(record)
      area = polygon.getInfo()['properties']['area'] #In acres
      roi = polygon.geometry()

      elevation = NASA_srtm.clip(roi).log().divide(10).clamp(0, 1).toFloat()

      landsat8_stats = landsat8_imagery.select(['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7','NBR','NDVI','GNDVI','EVI','NDMI','NDWI','NDBI','NDBaI','MNDWI']) \
      .reduceRegion(reducer=ee.Reducer.mean(), geometry=roi,scale=30)

      # landsat9_stats = landsat9_imagery.select(['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7','NBR','NDVI','GNDVI','EVI','NDMI','NDWI','NDBI','NDBaI','MNDWI']) \
      # .reduceRegion(reducer=ee.Reducer.mean(), geometry=roi,scale=30)

      sentinel_2_stats = sentinel_2_imagery.select(['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B9', 'B11', 'B12', 'NDVI', 'EVI', 'GNDVI','SAVI','VCI', 'TVI', 'BI', 'BI2', 'CI', 'CI1', 'SATVI', 'HVSI', 'SOCI', 'ASI', 'BSI', 'MSAVI']) \
      .addBands(elevation) \
      .reduceRegion(reducer=ee.Reducer.mean(), geometry=roi,scale=30)

      modis_lai_stats = modis_lai_imagery.select(['Lai']) \
      .reduceRegion(reducer=ee.Reducer.mean(), geometry=roi,scale=30)

      modis_vegetation_stats = modis_vegetation_imagery.select(['NDVI','EVI']) \
      .reduceRegion(reducer=ee.Reducer.mean(),geometry=roi,scale=30)


      USDA_stats = USDA_imagery.select(['ssm']) \
      .reduceRegion(reducer=ee.Reducer.mean(), geometry=roi,scale=30)

      indices = sentinel_2_stats.getInfo()
      modis_lai_indices = modis_lai_stats.getInfo()
      modis_vegetation_indices = modis_vegetation_stats.getInfo()
      usda_indices = USDA_stats.getInfo()
      landsat8_indices = landsat8_stats.getInfo()
      # landsat9_indices = landsat9_stats.getInfo()

      dictionary = {}
      dictionary['field_id'] = field_id
      dictionary['coordinates']= roi.getInfo()['coordinates'][0]
      dictionary['date'] = year['year']
      dictionary['Annual Sentinel 2 NDVI'] = indices.get('NDVI')
      dictionary['Annual Sentinel 2 EVI'] = indices.get('EVI')
      dictionary['Annual Sentinel 2 GNDVI'] = indices.get('GNDVI')
      dictionary['Annual Sentinel 2 SAVI'] = indices.get('SAVI')
      dictionary['Annual Sentinel 2 B1'] = indices.get('B1')
      dictionary['Annual Sentinel 2 B2'] = indices.get('B2')
      dictionary['Annual Sentinel 2 B3'] = indices.get('B3')
      dictionary['Annual Sentinel 2 B4'] = indices.get('B4')
      dictionary['Annual Sentinel 2 B5'] = indices.get('B5')
      dictionary['Annual Sentinel 2 B6'] = indices.get('B6')
      dictionary['Annual Sentinel 2 B7'] = indices.get('B7')
      dictionary['Annual Sentinel 2 B8'] = indices.get('B8')
      dictionary['Annual Sentinel 2 B9'] = indices.get('B9')
      dictionary['Annual Sentinel 2 B11'] = indices.get('B11')
      dictionary['Annual Sentinel 2 B12'] = indices.get('B12')

      dictionary['Annual Sentinel 2 VCI'] = indices.get('VCI')
      dictionary['Annual Sentinel 2 TVI'] = indices.get('TVI')
      dictionary['Annual Sentinel 2 BI'] = indices.get('BI')
      dictionary['Annual Sentinel 2 BI2'] = indices.get('BI2')
      dictionary['Annual Sentinel 2 CI'] = indices.get('CI')
      dictionary['Annual Sentinel 2 CI1'] = indices.get('CI1')
      dictionary['Annual Sentinel 2 SATVI'] = indices.get('SATVI')
      dictionary['Annual Sentinel 2 HVSI'] = indices.get('HVSI')
      dictionary['Annual Sentinel 2 SOCI'] = indices.get('SOCI')
      dictionary['Annual Sentinel 2 ASI'] = indices.get('ASI')
      dictionary['Annual Sentinel 2 BSI'] = indices.get('BSI')
      dictionary['Annual Sentinel 2 MSAVI'] = indices.get('MSAVI')

      dictionary['Annual Sentinel 2 Elevation'] = indices.get('elevation')

      dictionary['Annual SR8_B1'] = landsat8_indices.get('SR_B1')
      dictionary['Annual SR8_B2'] = landsat8_indices.get('SR_B2')
      dictionary['Annual SR8_B3'] = landsat8_indices.get('SR_B3')
      dictionary['Annual SR8_B4'] = landsat8_indices.get('SR_B4')
      dictionary['Annual SR8_B5'] = landsat8_indices.get('SR_B5')
      dictionary['Annual SR8_B6'] = landsat8_indices.get('SR_B6')
      dictionary['Annual SR8_B7'] = landsat8_indices.get('SR_B7')
      dictionary['Annual NBR_8'] = landsat8_indices.get('NBR')
      dictionary['Annual NDMI_8'] = landsat8_indices.get('NDMI')
      dictionary['Annual NDWI_8'] = landsat8_indices.get('NDWI')
      dictionary['Annual NDBI_8'] = landsat8_indices.get('NDBI')
      dictionary['Annual NDBaI_8'] = landsat8_indices.get('NDBaI')
      dictionary['Annual MNDWI_8'] = landsat8_indices.get('MNDWI')
      dictionary['Annual NDVI_8'] = landsat8_indices.get('NDVI')
      dictionary['Annual GNDVI_8'] = landsat8_indices.get('GNDVI')
      dictionary['Annual EVI_8'] = landsat8_indices.get('EVI')






      # dictionary['Annual SR9_B1'] = landsat9_indices.get('SR_B1')
      # dictionary['Annual SR9_B2'] = landsat9_indices.get('SR_B2')
      # dictionary['Annual SR9_B3'] = landsat9_indices.get('SR_B3')
      # dictionary['Annual SR9_B4'] = landsat9_indices.get('SR_B4')
      # dictionary['Annual SR9_B5'] = landsat9_indices.get('SR_B5')
      # dictionary['Annual SR9_B6'] = landsat9_indices.get('SR_B6')
      # dictionary['Annual SR9_B7'] = landsat9_indices.get('SR_B7')
      # dictionary['Annual NBR_9'] = landsat9_indices.get('NBR')
      # dictionary['Annual NDMI_9'] = landsat9_indices.get('NDMI')
      # dictionary['Annual NDWI_9'] = landsat9_indices.get('NDWI')
      # dictionary['Annual NDBI_9'] = landsat9_indices.get('NDBI')
      # dictionary['Annual NDBaI_9'] = landsat9_indices.get('NDBaI')
      # dictionary['Annual MNDWI_9'] = landsat9_indices.get('MNDWI')
      # dictionary['Annual NDVI_9'] = landsat9_indices.get('NDVI')
      # dictionary['Annual GNDVI_9'] = landsat9_indices.get('GNDVI')
      # dictionary['Annual EVI_9'] = landsat9_indices.get('EVI')

      dictionary['Annual MODIS LAI'] = modis_lai_indices.get('Lai')

      dictionary['Annual MODIS NDVI'] = normalize_modis_value(modis_vegetation_indices.get('NDVI'))
      dictionary['Annual MODIS EVI'] = normalize_modis_value(modis_vegetation_indices.get('EVI'))

      dictionary['Annual MOISTURE'] = usda_indices.get('ssm')

      dictionary['Area'] = area

      feature_data.append(dictionary)
      field_id = field_id + 1
    df = pd.DataFrame(feature_data)
    feature_data = []
    # Save the DataFrame as a CSV file
    df.to_csv(f'Louisiana_4Y_Records_{index}.csv', index=False)

    # Download the CSV file in Google Colab
    from google.colab import files
    files.download(f'Louisiana_4Y_Records_{index}.csv')
    index = index + 1
    print("Field ID with Year", field_id, start_date)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Field ID with Year 1481 2018-01-01


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

Field ID with Year 8881 2018-01-01


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Field ID with Year 10361 2018-01-01


In [ ]:
len(feature_data)

17

In [ ]:
df = pd.DataFrame(feature_data)

,field_id,coordinates,date,Annual Sentinel 2 NDVI,Annual Sentinel 2 EVI,Annual Sentinel 2 GNDVI,Annual Sentinel 2 SAVI,Annual Sentinel 2 B1,Annual Sentinel 2 B2,Annual Sentinel 2 B3,...,Annual NDBaI_8,Annual MNDWI_8,Annual NDVI_8,Annual GNDVI_8,Annual EVI_8,Annual MODIS LAI,Annual MODIS NDVI,Annual MODIS EVI,Annual MOISTURE,Area


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
feature_weights = {
        'field_id': 0,
        'coorrdinates': 0,
        'date': 0,
        'Annual NDVI': 0.25,
        'Annual EVI': 0.2,
        'Annual GNDVI': 0.15,
        'Annual SAVI': 0.15,
        'Annual B1': 0.05,
        'Annual B2': 0.05,
        'Annual B3': 0.05,
        'Annual B4': 0.05,
        'Annual B5': 0.05,
        'Annual B6': 0.05,
        'Annual B7': 0.05,
        'Annual B8': 0.05,
        'Annual B9': 0.05,
        'Annual B11': 0.05,
        'Annual B12': 0.05,
        'Annual SR8_B1': 0.05,
        'Annual SR8_B2': 0.05,
        'Annual SR8_B3': 0.05,
        'Annual SR8_B4': 0.05,
        'Annual SR8_B5': 0.05,
        'Annual SR8_B6': 0.05,
        'Annual SR8_B7': 0.05,
        'Annual NBR_8': 0.03,
        'Annual NDVI_8':0.25,
        'Annual EVI_8':0.2,
        'Annual GNDVI_8':0.15,
        'Annual NDMI_8': 0.04,
        'Annual NDWI_8': 0.04,
        'Annual NDBI_8': 0.03,
        'Annual NDBaI_8': 0.02,
        'Annual MNDWI_8': 0.03,
        'Annual SR9_B1': 0.05,
        'Annual SR9_B2': 0.05,
        'Annual SR9_B3': 0.05,
        'Annual SR9_B4': 0.05,
        'Annual SR9_B5': 0.05,
        'Annual SR9_B6': 0.05,
        'Annual SR9_B7': 0.05,
        'Annual NBR_9': 0.03,
        'Annual NDVI_9':0.25,
        'Annual EVI_9':0.2,
        'Annual GNDVI_9':0.15,
        'Annual NDMI_9': 0.04,
        'Annual NDWI_9': 0.04,
        'Annual NDBI_9': 0.03,
        'Annual NDBaI_9': 0.02,
        'Annual MNDWI_9': 0.03,
        'Annual LAI': 0.2,
        'Annual MOISTURE': 0.1,
        'Area': 0.3,
        'Elevation': 0.1,
        'Annual VCI': 0.1,
        'Annual TVI': 0.1,
        'Annual BI': 0.05,
        'Annual BI2': 0.05,
        'Annual CI': 0.05,
        'Annual CI1': 0.05,
        'Annual SATVI': 0.1,
        'Annual HVSI': 0.1,
        'Annual SOCI': 0.05,
        'Annual ASI': 0.05,
        'Annual BSI': 0.05,
        'Annual MSAVI': 0.1,
        'Annual MODIS NDVI':0.25,
        'Annual MODIS EVI':0.2
    }

In [ ]:
# Create a feature collection for the specified state (Florida in this case)
florida = ee.FeatureCollection('TIGER/2018/States') \
    .filter(ee.Filter.eq('STUSPS', 'TX'))
roi = florida.geometry()
# Print the feature collection (optional)


In [ ]:
start_date = '2023-01-01'
end_date = '2023-12-01'
yield_annually = 30

sentinel2_data = sentinel2 \
.filterDate(start_date, end_date)

landsat8_data = landsat8 \
.filterDate(start_date, end_date)

landsat9_data = landsat9 \
.filterDate(start_date, end_date)

modis_lai = modis_lai_data \
.filterDate(start_date, end_date)

modis_vegetation = modis_vegetation_data \
.filterDate(start_date, end_date)

USDA_SoilData_data = USDA_SoilData \
.filterDate(start_date, end_date)

landsat8_imagery = landsat8_data.mosaic().select(['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7','NBR','NDVI','GNDVI','EVI','NDMI','NDWI','NDBI','NDBaI','MNDWI'])
landsat9_imagery = landsat9_data.mosaic().select(['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7','NBR','NDVI','GNDVI','EVI','NDMI','NDWI','NDBI','NDBaI','MNDWI'])
sentinel_2_imagery = sentinel2_data.mosaic().select(['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B9', 'B11', 'B12', 'NDVI', 'EVI', 'GNDVI','SAVI','VCI', 'TVI', 'BI', 'BI2', 'CI', 'CI1', 'SATVI', 'HVSI', 'SOCI', 'ASI', 'BSI', 'MSAVI'])
modis_lai_imagery = modis_lai.mosaic().select(['Lai'])
modis_vegetation_imagery = modis_vegetation.mosaic().select('NDVI','EVI')
USDA_imagery = USDA_SoilData_data.mosaic().select(['ssm'])


elevation = NASA_srtm.clip(roi).log().divide(10).clamp(0, 1).toFloat()

landsat8_stats = landsat8_imagery.select(['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7','NBR','NDVI','GNDVI','EVI','NDMI','NDWI','NDBI','NDBaI','MNDWI']) \
.reduceRegion(reducer=ee.Reducer.mean(), geometry=roi,scale=100,maxPixels=1e10)

landsat9_stats = landsat9_imagery.select(['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7','NBR','NDVI','GNDVI','EVI','NDMI','NDWI','NDBI','NDBaI','MNDWI']) \
.reduceRegion(reducer=ee.Reducer.mean(), geometry=roi,scale=100,maxPixels=1e10)

sentinel_2_stats = sentinel_2_imagery.select(['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B9', 'B11', 'B12', 'NDVI', 'EVI', 'GNDVI','SAVI','VCI', 'TVI', 'BI', 'BI2', 'CI', 'CI1', 'SATVI', 'HVSI', 'SOCI', 'ASI', 'BSI', 'MSAVI']) \
.addBands(elevation) \
.reduceRegion(reducer=ee.Reducer.mean(), geometry=roi,scale=100,maxPixels=1e10)

modis_lai_stats = modis_lai_imagery.select(['Lai']) \
.reduceRegion(reducer=ee.Reducer.mean(), geometry=roi,scale=100,maxPixels=1e10)

modis_vegetation_stats = modis_vegetation_imagery.select(['NDVI','EVI']) \
.reduceRegion(reducer=ee.Reducer.mean(),geometry=roi,scale=100,maxPixels=1e10)


USDA_stats = USDA_imagery.select(['ssm']) \
.reduceRegion(reducer=ee.Reducer.mean(), geometry=roi,scale=100,maxPixels=1e10)

indices = sentinel_2_stats.getInfo()
modis_lai_indices = modis_lai_stats.getInfo()
modis_vegetation_indices = modis_vegetation_stats.getInfo()
usda_indices = USDA_stats.getInfo()
landsat8_indices = landsat8_stats.getInfo()
landsat9_indices = landsat9_stats.getInfo()

dictionary = {}
dictionary['field_id'] = field_id
dictionary['coorrdinates']= roi.getInfo()['coordinates'][0]
dictionary['date'] = year['year']
dictionary['Annual NDVI'] = indices.get('NDVI')
dictionary['Annual EVI'] = indices.get('EVI')
dictionary['Annual GNDVI'] = indices.get('GNDVI')
dictionary['Annual SAVI'] = indices.get('SAVI')
dictionary['Annual B1'] = indices.get('B1')
dictionary['Annual B2'] = indices.get('B2')
dictionary['Annual B3'] = indices.get('B3')
dictionary['Annual B4'] = indices.get('B4')
dictionary['Annual B5'] = indices.get('B5')
dictionary['Annual B6'] = indices.get('B6')
dictionary['Annual B7'] = indices.get('B7')
dictionary['Annual B8'] = indices.get('B8')
dictionary['Annual B9'] = indices.get('B9')
dictionary['Annual B11'] = indices.get('B11')
dictionary['Annual B12'] = indices.get('B12')

dictionary['Annual VCI'] = indices.get('VCI')
dictionary['Annual TVI'] = indices.get('TVI')
dictionary['Annual BI'] = indices.get('BI')
dictionary['Annual BI2'] = indices.get('BI2')
dictionary['Annual CI'] = indices.get('CI')
dictionary['Annual CI1'] = indices.get('CI1')
dictionary['Annual SATVI'] = indices.get('SATVI')
dictionary['Annual HVSI'] = indices.get('HVSI')
dictionary['Annual SOCI'] = indices.get('SOCI')
dictionary['Annual ASI'] = indices.get('ASI')
dictionary['Annual BSI'] = indices.get('BSI')
dictionary['Annual MSAVI'] = indices.get('MSAVI')

dictionary['Elevation'] = indices.get('elevation')

dictionary['Annual SR8_B1'] = landsat8_indices.get('SR_B1')
dictionary['Annual SR8_B2'] = landsat8_indices.get('SR_B2')
dictionary['Annual SR8_B3'] = landsat8_indices.get('SR_B3')
dictionary['Annual SR8_B4'] = landsat8_indices.get('SR_B4')
dictionary['Annual SR8_B5'] = landsat8_indices.get('SR_B5')
dictionary['Annual SR8_B6'] = landsat8_indices.get('SR_B6')
dictionary['Annual SR8_B7'] = landsat8_indices.get('SR_B7')
dictionary['Annual NBR_8'] = landsat8_indices.get('NBR')
dictionary['Annual NDMI_8'] = landsat8_indices.get('NDMI')
dictionary['Annual NDWI_8'] = landsat8_indices.get('NDWI')
dictionary['Annual NDBI_8'] = landsat8_indices.get('NDBI')
dictionary['Annual NDBaI_8'] = landsat8_indices.get('NDBaI')
dictionary['Annual MNDWI_8'] = landsat8_indices.get('MNDWI')
dictionary['Annual NDVI_8'] = landsat8_indices.get('NDVI')
dictionary['Annual GNDVI_8'] = landsat8_indices.get('GNDVI')
dictionary['Annual EVI_8'] = landsat8_indices.get('EVI')






dictionary['Annual SR9_B1'] = landsat9_indices.get('SR_B1')
dictionary['Annual SR9_B2'] = landsat9_indices.get('SR_B2')
dictionary['Annual SR9_B3'] = landsat9_indices.get('SR_B3')
dictionary['Annual SR9_B4'] = landsat9_indices.get('SR_B4')
dictionary['Annual SR9_B5'] = landsat9_indices.get('SR_B5')
dictionary['Annual SR9_B6'] = landsat9_indices.get('SR_B6')
dictionary['Annual SR9_B7'] = landsat9_indices.get('SR_B7')
dictionary['Annual NBR_9'] = landsat9_indices.get('NBR')
dictionary['Annual NDMI_9'] = landsat9_indices.get('NDMI')
dictionary['Annual NDWI_9'] = landsat9_indices.get('NDWI')
dictionary['Annual NDBI_9'] = landsat9_indices.get('NDBI')
dictionary['Annual NDBaI_9'] = landsat9_indices.get('NDBaI')
dictionary['Annual MNDWI_9'] = landsat9_indices.get('MNDWI')
dictionary['Annual NDVI_9'] = landsat9_indices.get('NDVI')
dictionary['Annual GNDVI_9'] = landsat9_indices.get('GNDVI')
dictionary['Annual EVI_9'] = landsat9_indices.get('EVI')

dictionary['Annual MODIS LAI'] = modis_lai_indices.get('Lai')

dictionary['Annual MODIS NDVI'] = normalize_modis_value(modis_vegetation_indices.get('NDVI'))
dictionary['Annual MODIS EVI'] = normalize_modis_value(modis_vegetation_indices.get('EVI'))

dictionary['Annual MOISTURE'] = usda_indices.get('ssm')

dictionary['Area'] = area
dictionary

In [ ]:
state_level_indices = []

In [ ]:
start_date = '2023-01-01'
end_date = '2023-12-31'
roi = sugarcane_tx.geometry()

sentinel2_data = sentinel2 \
.filterDate(start_date, end_date)


landsat8_data = landsat8 \
.filterDate(start_date, end_date)


modis_lai = modis_lai_data \
.filterDate(start_date, end_date)

modis_vegetation = modis_vegetation_data \
.filterDate(start_date, end_date)

USDA_SoilData_data = USDA_SoilData \
.filterDate(start_date, end_date)

landsat8_imagery = landsat8_data.mosaic().select(['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7','NBR','NDVI','GNDVI','EVI','NDMI','NDWI','NDBI','NDBaI','MNDWI'])
sentinel_2_imagery = sentinel2_data.median().select(['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B9', 'B11', 'B12', 'NDVI', 'EVI', 'GNDVI','SAVI','VCI', 'TVI', 'BI', 'BI2', 'CI', 'CI1', 'SATVI', 'HVSI', 'SOCI', 'ASI', 'BSI', 'MSAVI'])
modis_lai_imagery = modis_lai.mosaic().select(['Lai'])
modis_vegetation_imagery = modis_vegetation.mosaic().select('NDVI','EVI')
USDA_imagery = USDA_SoilData_data.mosaic().select(['ssm'])




elevation = NASA_srtm.clip(roi).log().divide(10).clamp(0, 1).toFloat()

landsat8_stats = landsat8_imagery.select(['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7','NBR','NDVI','GNDVI','EVI','NDMI','NDWI','NDBI','NDBaI','MNDWI']) \
.reduceRegion(reducer=ee.Reducer.mean(), geometry=roi,scale=250, maxPixels=1e13 )


sentinel_2_stats = sentinel_2_imagery.select(['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B9', 'B11', 'B12', 'NDVI', 'EVI', 'GNDVI','SAVI','VCI', 'TVI', 'BI', 'BI2', 'CI', 'CI1', 'SATVI', 'HVSI', 'SOCI', 'ASI', 'BSI', 'MSAVI']) \
.addBands(elevation) \
.reduceRegion(reducer=ee.Reducer.mean(), geometry=roi,scale=250, maxPixels=1e13 )

modis_lai_stats = modis_lai_imagery.select(['Lai']) \
.reduceRegion(reducer=ee.Reducer.mean(), geometry=roi,scale=250, maxPixels=1e13 )

modis_vegetation_stats = modis_vegetation_imagery.select(['NDVI','EVI']) \
.reduceRegion(reducer=ee.Reducer.mean(),geometry=roi,scale=250, maxPixels=1e13 )


USDA_stats = USDA_imagery.select(['ssm']) \
.reduceRegion(reducer=ee.Reducer.mean(), geometry=roi,scale=250, maxPixels=1e13 )

indices = sentinel_2_stats.getInfo()
print(indices)
modis_lai_indices = modis_lai_stats.getInfo()
modis_vegetation_indices = modis_vegetation_stats.getInfo()
usda_indices = USDA_stats.getInfo()
landsat8_indices = landsat8_stats.getInfo()


dictionary = {}
dictionary['date'] = 2022
dictionary['Annual Sentinel 2 NDVI'] = indices.get('NDVI')
dictionary['Annual Sentinel 2 EVI'] = indices.get('EVI')
dictionary['Annual Sentinel 2 GNDVI'] = indices.get('GNDVI')
dictionary['Annual Sentinel 2 SAVI'] = indices.get('SAVI')
dictionary['Annual Sentinel 2 B1'] = indices.get('B1')
dictionary['Annual Sentinel 2 B2'] = indices.get('B2')
dictionary['Annual Sentinel 2 B3'] = indices.get('B3')
dictionary['Annual Sentinel 2 B4'] = indices.get('B4')
dictionary['Annual Sentinel 2 B5'] = indices.get('B5')
dictionary['Annual Sentinel 2 B6'] = indices.get('B6')
dictionary['Annual Sentinel 2 B7'] = indices.get('B7')
dictionary['Annual Sentinel 2 B8'] = indices.get('B8')
dictionary['Annual Sentinel 2 B9'] = indices.get('B9')
dictionary['Annual Sentinel 2 B11'] = indices.get('B11')
dictionary['Annual Sentinel 2 B12'] = indices.get('B12')

dictionary['Annual Sentinel 2 VCI'] = indices.get('VCI')
dictionary['Annual Sentinel 2 TVI'] = indices.get('TVI')
dictionary['Annual Sentinel 2 BI'] = indices.get('BI')
dictionary['Annual Sentinel 2 BI2'] = indices.get('BI2')
dictionary['Annual Sentinel 2 CI'] = indices.get('CI')
dictionary['Annual Sentinel 2 CI1'] = indices.get('CI1')
dictionary['Annual Sentinel 2 SATVI'] = indices.get('SATVI')
dictionary['Annual Sentinel 2 HVSI'] = indices.get('HVSI')
dictionary['Annual Sentinel 2 SOCI'] = indices.get('SOCI')
dictionary['Annual Sentinel 2 ASI'] = indices.get('ASI')
dictionary['Annual Sentinel 2 BSI'] = indices.get('BSI')
dictionary['Annual Sentinel 2 MSAVI'] = indices.get('MSAVI')

dictionary['Annual Sentinel 2 Elevation'] = indices.get('elevation')

dictionary['Annual SR8_B1'] = landsat8_indices.get('SR_B1')
dictionary['Annual SR8_B2'] = landsat8_indices.get('SR_B2')
dictionary['Annual SR8_B3'] = landsat8_indices.get('SR_B3')
dictionary['Annual SR8_B4'] = landsat8_indices.get('SR_B4')
dictionary['Annual SR8_B5'] = landsat8_indices.get('SR_B5')
dictionary['Annual SR8_B6'] = landsat8_indices.get('SR_B6')
dictionary['Annual SR8_B7'] = landsat8_indices.get('SR_B7')
dictionary['Annual NBR_8'] = landsat8_indices.get('NBR')
dictionary['Annual NDMI_8'] = landsat8_indices.get('NDMI')
dictionary['Annual NDWI_8'] = landsat8_indices.get('NDWI')
dictionary['Annual NDBI_8'] = landsat8_indices.get('NDBI')
dictionary['Annual NDBaI_8'] = landsat8_indices.get('NDBaI')
dictionary['Annual MNDWI_8'] = landsat8_indices.get('MNDWI')
dictionary['Annual NDVI_8'] = landsat8_indices.get('NDVI')
dictionary['Annual GNDVI_8'] = landsat8_indices.get('GNDVI')
dictionary['Annual EVI_8'] = landsat8_indices.get('EVI')




dictionary['Annual MODIS LAI'] = modis_lai_indices.get('Lai')

dictionary['Annual MODIS NDVI'] = normalize_modis_value(modis_vegetation_indices.get('NDVI'))
dictionary['Annual MODIS EVI'] = normalize_modis_value(modis_vegetation_indices.get('EVI'))

dictionary['Annual MOISTURE'] = usda_indices.get('ssm')

state_level_indices.append(dictionary)




{'ASI': -33.141419448358, 'B1': 2181.1900563408553, 'B11': 3930.6317490442957, 'B12': 3255.653025065278, 'B2': 2178.0151335212477, 'B3': 2441.4233780682403, 'B4': 2498.4411136350227, 'B5': 2932.567160414452, 'B6': 4396.798493781644, 'B7': 4933.907149496655, 'B8': 4894.803470349268, 'B9': 5205.28617250264, 'BI': 5475.726584397714, 'BI2': 3588.1626707813703, 'BSI': -0.44139644958111385, 'CI': 0.44137244121683167, 'CI1': -0.2976512254352804, 'EVI': 0.8229620973295511, 'GNDVI': 0.20058359337936868, 'HVSI': 1271.380613272748, 'MSAVI': 0.2976179695026252, 'NDVI': 0.1771467334632989, 'SATVI': 0.1770519651038373, 'SAVI': 0.26577299287973455, 'SOCI': -205.95697529302092, 'TVI': 0.0885675166539727, 'VCI': 22.145186315503103, 'elevation': 0.28191974894330096}


In [ ]:
df = pd.DataFrame(state_level_indices)
df

,date,Annual Sentinel 2 NDVI,Annual Sentinel 2 EVI,Annual Sentinel 2 GNDVI,Annual Sentinel 2 SAVI,Annual Sentinel 2 B1,Annual Sentinel 2 B2,Annual Sentinel 2 B3,Annual Sentinel 2 B4,Annual Sentinel 2 B5,...,Annual NDBI_8,Annual NDBaI_8,Annual MNDWI_8,Annual NDVI_8,Annual GNDVI_8,Annual EVI_8,Annual MODIS LAI,Annual MODIS NDVI,Annual MODIS EVI,Annual MOISTURE
0,2018,0.510696,1.225484,0.514664,0.765934,293.707347,489.203926,797.569324,795.255244,1232.739087,...,-0.055021,0.097596,-0.195051,0.242145,0.246671,1.482898,2.340763,-0.443191,-0.497763,4.638592
1,2019,0.575887,1.824435,0.521776,0.863708,612.372038,683.985897,1004.335344,835.873543,1379.315789,...,-0.045450,0.091104,-0.184301,0.224064,0.226561,1.200796,3.505672,-0.008984,-0.244918,4.671412
2,2021,0.609611,1.869806,0.546004,0.914302,564.028831,650.920773,987.674162,811.473844,1367.469471,...,-0.104146,0.111388,-0.169284,0.272159,0.267744,1.507070,6.618510,0.156520,-0.130540,7.370429
3,2022,0.177147,0.822962,0.200584,0.265773,2181.190056,2178.015134,2441.423378,2498.441114,2932.567160,...,-0.056943,0.101082,-0.174731,0.233847,0.228738,1.744777,3.853830,-0.094747,-0.335306,1.996733


In [ ]:

# Save the DataFrame as a CSV file
df.to_csv('State Level.csv', index=False)

# Download the CSV file in Google Colab
from google.colab import files
files.download('State Level.csv')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>